# Draw Keyjoint

## PyTorch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import csv
import json
import os
import pandas as pd
import numpy as np
import torchvision
from torchvision.utils import draw_keypoints, make_grid, draw_bounding_boxes
from PIL import Image
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Successfully connected to: {device}")
model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=True)
model.to(device)
model.eval()

In [ ]:
skeleton_edges = [
    [0, 1], [0, 2], [1, 2], [1, 3], [2, 4], [3, 5], [4, 6], [5,6], [5, 7],
    [5, 11], [6, 12], [6, 8], [7, 9], [8, 10], [11, 12], [13, 11],
    [14, 12], [15, 13], [16, 14]
]
parts = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]
column_names = []
for part in parts:
    column_names.extend([f"{part}_x", f"{part}_y", f"{part}_v"])
keypoint_data = []

In [ ]:
def get_keypoints(image_id, image_dir, coco_dict, output_img_dir, csv_filename):
  img_info = next(item for item in coco_dict['images'] if item['id'] == image_id)
  file_name = img_info['file_name']

  ann_info = [item for item in coco_dict['annotations'] if item['image_id'] == image_id]

  image_path = os.path.join(image_dir, file_name)

  img = Image.open(image_path).convert("RGB")
  img_tensor = torch.as_tensor(np.array(img)).permute(2, 0, 1)

  bbox = []
  cat_id = 0
  for ann in ann_info:
    x,y,w,h = ann['bbox']
    cat_id = ann['category_id']
    bbox.append([x,y,x+w,y+h])

  bbox_tensor = torch.tensor(bbox, dtype=torch.float32)
  img_with_boxes = draw_bounding_boxes(img_tensor, bbox_tensor, colors='red',width=2)
  img_for_drawing = F.to_tensor(img).unsqueeze(0).to(device)

  with torch.no_grad():
      output = model(img_for_drawing)[0]

  keypoints = output['keypoints']
  keypoints_scores = output['keypoints_scores']

  #Skip if empty result
  if (keypoints.shape[0] == 0) : return
  scores = output['scores']

  # 90% con7firm result
  high_score_kp = keypoints[scores > 0.9]
  high_score_kp_scores = keypoints_scores[scores > 0.9]

  #debug
  print(scores)
  print(high_score_kp)

  if high_score_kp.shape[0] == 0: return

  #if more than one
  selected_kp = high_score_kp[0]
  selected_kp_scores = high_score_kp_scores[0]
  print(selected_kp_scores)

  kp_filtered = selected_kp.clone()
  
  # ====== Include dah visibility to 0.0 instead of null ======
  for i in range(17):
    if selected_kp_scores[i] < 0.0:
        kp_filtered[i, :] = torch.tensor([0.0, 0.0, 0.0], device=kp_filtered.device)

  #store csv
  data_row = kp_filtered[:, :3].reshape(-1).cpu().numpy()
  full_row = [image_id, cat_id] + data_row.tolist()

  file_exists = os.path.isfile(csv_filename)

  with open(csv_filename, mode='a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists:
      writer.writerow(['image_id', 'cat_id'] + column_names)
    writer.writerow(full_row)

  #draw kp on image
  if len(selected_kp[0]) > 0:
    pred_keypoints, visibility = kp_filtered.split([2, 1], dim=-1)
    visibility = visibility.bool()

    annotated_img = draw_keypoints(
        img_with_boxes,
        pred_keypoints.unsqueeze(0),
        visibility=visibility.unsqueeze(0),
        colors='red',
        radius=4,
        connectivity=skeleton_edges
    )

  output_filename = os.path.join(output_img_dir, "keyjoints_" + file_name)
  torchvision.io.write_png((annotated_img).to(torch.uint8), output_filename)

In [ ]:
#####################
# Code by Dennis
#####################
base_path = '/content/drive/MyDrive/Sitting Posture dataset'
splits = ['train', 'valid', 'test']

for split in splits:
    print(f"\n========== Processing {split.upper()} folder ==========")

    image_dir = os.path.join(base_path, split)
    json_path = os.path.join(image_dir, '_annotations.coco.json')

    csv_output_name = f"keypoint_{split}_data.csv"

    output_img_dir = f"keyjoint_{split}_dataset"
    if not os.path.exists(output_img_dir):
        os.makedirs(output_img_dir)
        print(f"Created image directory: {output_img_dir}")

    with open(json_path, 'r') as f:
        current_coco_data = json.load(f)

    image_ids = [img['id'] for img in current_coco_data['images']]

    for i, img_id in enumerate(image_ids):
        get_keypoints(img_id, image_dir, current_coco_data, output_img_dir, csv_output_name)

        if (i + 1) % 100 == 0 or (i + 1) == len(image_ids):
            print(f"Processed {i + 1} / {len(image_ids)} images...")

    print(f"Finished generating {csv_output_name} for {split} split.")
#####################
# End
#####################